# Platform & Workspace

**Module 1 | Fundamentals Training**

We explore the Databricks Lakehouse platform — architecture, workspace components, compute types, and the basics of Unity Catalog. We also cover **cost management** principles and learn to work efficiently with the **Databricks GUI**: notebooks, markdown, mixing SQL and Python, simple visualizations, and the AI Assistant.


## Learning Objectives

After completing this module you will be able to:

- **Describe** the Databricks Lakehouse architecture — how it combines Data Lake and Data Warehouse benefits
- **Navigate** the Databricks workspace: notebooks, clusters, DBFS, Catalog Explorer, Git Folders
- **Explain** the difference between All-Purpose clusters and Job clusters and their cost implications
- **Understand** autoscaling and how it impacts costs
- **Apply** basic cost management principles: choosing the right cluster type, sizing, auto-termination
- **Use** the GUI effectively: write markdown, mix SQL and Python in the same notebook
- **Create** simple visualizations directly in the notebook GUI
- **Understand** how the Databricks AI Assistant supports daily work in notebooks
- **Describe** Unity Catalog's 3-level namespace: `catalog.schema.table`


## Databricks Platform Elements

Key elements of the Databricks platform: Workspace, Catalog Explorer, Git Folders, Volumes. Understanding the platform structure is the foundation of a Data Engineer's work.

---

### Per-user Isolation

Run the initialization script for per-user catalog and schema isolation:


In [0]:
%run ../setup/00_setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re

# Display user context (variables from 00_setup)
print(f"Catalog: {CATALOG}")
print(f"Schema Bronze: {BRONZE_SCHEMA}")
print(f"Schema Silver: {SILVER_SCHEMA}")
print(f"Schema Gold: {GOLD_SCHEMA}")
print(f"User: {raw_user}")

### Comparison of Traditional Architecture vs Lakehouse

**Objective:** Visualize differences between traditional approach (Data Lake + Data Warehouse) and Lakehouse.

![alt text](../../assets/images/4c9090bd82f2475c810bafde13f978e0.png)

**Traditional Architecture:**

<img src="../../assets/images/49f3830d3784442ea5582bc82e6fb89c.png" width="800">

**Lakehouse Benefits:**
- Single copy of data (single source of truth)
- Lower storage costs
- Elimination of synchronization latency
- Common governance for all use cases

### Databricks Platform Elements

**Theoretical Introduction:**

The Databricks platform consists of several key components that together create a complete environment for working with data in the Lakehouse architecture.

**Key Components:**
- **Workspace**: Working environment containing notebooks, experiments, folders, and resources
- **Catalog Explorer**: Interface for managing catalogs, schemas, tables, and views
- **Git Folders (formerly Repos)**: Git integration for versioning notebooks and code
- **Volumes**: Management of unstructured files (images, models, artifacts)
- **DBFS (Databricks File System)** *(legacy — use Volumes instead)*: Virtual file system over cloud storage

**Practical Application:**
- Workspace organizes projects and team collaboration
- Catalog Explorer enables data exploration and governance
- Git Folders integrates development workflow with Git

### Workspace Exploration

#### Example: Workspace Exploration

**Objective:** Familiarize with Databricks Workspace interface

**Workspace Elements:**
1. **Sidebar** (left side):
   - Workspace: Folders and notebooks
   - Git Folders: Git Integration
   - Compute: Cluster management
   - Workflows: Lakeflow Jobs
   - Catalog: Unity Catalog explorer

2. **Main Panel**: Notebook editor or details view

3. **Top Bar**: Quick access to compute, account, help

**Navigation Instructions:**
- Use the left menu to switch between sections
- In the Catalog section, you can browse catalogs, schemas, and tables
- In the Compute section, you manage Spark clusters

#### Databricks Workspace UI

<img src="../../assets/images/848bc3658ab44bb09f586bd2b1f4231e.png" width="800">
> ```

### Catalog Explorer - Unity Catalog Structure

#### Example: Catalog Explorer

**Objective:** Understand object hierarchy in Unity Catalog

#### Catalog Explorer Screenshot

<img src="../../assets/images/32356ba877a74bfe87feeb6d6ee93a46.png" width="800">


In [0]:
# Display current catalog and schema
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
current_schema = spark.sql("SELECT current_schema()").collect()[0][0]

print(f"Current catalog: {current_catalog}")
print(f"Current schema: {current_schema}")

**Unity Catalog Hierarchy:**

<img src="../../assets/images/cddc09f5ffc5482aa3063a13a7c4f927.png" width="800">

> **3-level namespace:** `catalog.schema.object` — e.g., `prod.gold.sales_summary`

### Browsing Catalogs and Schemas

#### Example: Browsing Catalogs and Schemas

**Objective:** Programmatic listing of objects in Unity Catalog

In [0]:
# List of all catalogs available to the user
catalogs_df = spark.sql("SHOW CATALOGS")
display(catalogs_df)

In [0]:
# List of schemas in the current catalog
schemas_df = spark.sql(f"SHOW SCHEMAS IN {CATALOG}")
display(schemas_df)

### Git Folders and Git Integration

In practice, working with code in Databricks should be based on **Git Folders** (formerly Repos), not single, orphaned notebooks in Workspace.

Typical workflow:

1. **Create Git Folder** in Databricks: `Workspace → Git Folders → Add Repo`.
2. **Connect to Git** (GitHub / Azure DevOps / other).
3. Work on **feature branches** (e.g., `feature/cleaning-module`).
4. Regularly:
   - commit and push changes from Databricks to remote repo,
   - create PR and merge to main/dev.

Best Practices:

- One repo per project/domain (e.g., `databricks-dea-training`).
- Do not work in **Workspace root** – always in **Git Folders**.
- Training notebooks, test data, and README can be in one repo.

### Volumes vs DBFS

Where should you store files? In new Unity Catalog-based workspaces, **Volumes** are the preferred location.

- `dbfs:/` is treated as a **legacy** layer or auxiliary area.
- `/Volumes/catalog.schema.volume_name` is a fully managed, UC-controlled data area (permissions, audit, lineage).

Volume Definition Example (SQL):

```sql
CREATE VOLUME IF NOT EXISTS ${catalog}.${schema}.training_volume
COMMENT 'Workspace for training purposes';
```

Usage Example in PySpark:

```python
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

volume_path = f"/Volumes/{catalog}/{schema}/training_volume"
display(dbutils.fs.ls(volume_path))
```

### SQL Warehouse

A SQL engine optimized for BI and ad-hoc analytics, an alternative to notebook clusters.

When to use:
- Reporting in Power BI / other BI tools.
- Business analysts / power users working mainly in SQL.
- Interactive dashboards and ad-hoc queries to **Gold** layer.

Differences from all-purpose cluster:
- Billing based on **DBU SQL** (different rates).
- Automatic provisioning / scaling.
- Isolation of BI workload from engineering clusters.

---

## Compute Resources

Types of compute resources in Databricks: All-Purpose Clusters, Job Clusters, SQL Warehouses. Choosing the right compute directly impacts costs.

---

### The Real Question: How Much Will This Cost?

As a Data Engineer, you'll be asked: *"Why is our Databricks bill so high?"*

Understanding compute options is essential for cost control.

### Compute Options Comparison

| Type | Startup Time | Cost Model | Best For |
|------|--------------|------------|----------|
| **All-Purpose Cluster** | 3-5 min | Per-minute (running) | Interactive development, exploration |
| **Job Cluster** | 3-5 min | Per-minute (only during job) | Scheduled production jobs |
| **Serverless** | <10 sec | Per-query DBUs | Ad-hoc queries, variable workloads |
| **SQL Warehouse** | 0 (Serverless) or 3-5 min | Per-query DBUs | BI tools, SQL analysts |

### Cost Optimization Strategies

**1. Right-size clusters:**
- Development: 2-4 workers, smallest instance type
- Production: Autoscaling 2-10 workers based on workload

**2. Use Spot/Preemptible instances:**
- 60-80% cost savings for workers
- Driver on on-demand (stability)
- Trade-off: Job may be interrupted

**3. Photon Engine:**
- 2-3x faster for aggregations/joins
- ~2x DBU cost, but finishes faster = often cheaper
- Enable for: large scans, aggregations, joins
- Skip for: simple transformations, ML training

**4. Cluster policies:**
- Enforce maximum worker count
- Require autoscaling
- Set auto-termination (e.g., 30 min idle)

### Decision Tree: Which Compute to Use?

<img src="../../assets/images/f98e20f71ec541eb9b206877f1da98b5.png" width="800">

---

---
### Serverless Compute — the new default in Databricks

> **Since 2024, Serverless is the default compute type** for new notebooks and jobs in most regions.

**What is Serverless?**
- Databricks manages the infrastructure — **you don't need to configure a cluster**
- Starts in **< 10 seconds** (vs 3-5 minutes for a classic cluster)
- You pay only for actually consumed resources (seconds of execution, not cluster uptime)
- Automatic resource scaling in the background — invisible to the user

**Serverless models in Databricks:**

| Context | Serverless compute |
|----------|-------------------|
| Notebooks (Serverless clusters) | Cluster starts in < 10s, stops when the session ends |
| Workflow Jobs (Serverless jobs) | Each run = its own isolated compute |
| SQL Warehouse (Serverless) | Fastest type — dedicated to SQL / BI |
| Delta Live Tables (Serverless) | DLT pipelines without managing compute |

**When NOT to use Serverless?**
- Large ML workloads with GPU (e.g., model fine-tuning) — a dedicated cluster with GPU is needed here
- Workloads requiring specific libraries installed at system level (custom init scripts)
- Extremely large shuffles (100+ GB) — a classic cluster may be more efficient

> **How to enable Serverless for a notebook:** In the top-right corner, select "Connect to compute" → "Serverless"


---
### SQL Warehouse — dedicated compute for SQL and BI

**SQL Warehouse** is a separate compute type, decoupled from notebook clusters. Designed specifically for:
- Analysts using the SQL Editor
- BI tools: Tableau, Power BI, Looker
- AI/BI Dashboards and Genie in Databricks

**SQL Warehouse types:**

| Type | Startup | Cost | When to use |
|------|---------|------|------------|
| **Serverless** | < 3 sec | Higher DBU/hr, but only query time | Interactive SQL work, variable workloads |
| **Pro** | 3-5 min | Medium, rich features | Ad-hoc analytics, advanced workloads |
| **Classic** | 3-5 min | Lowest, basic features | Simple SQL queries, CI/CD query testing |

**How to create a SQL Warehouse:**
1. Left menu → **SQL Warehouses**
2. Click **Create SQL Warehouse**
3. Choose type: **Serverless** (recommended for training)
4. Cluster Size: `X-Small` (sufficient for demo)
5. Auto Stop: `10 minutes` (cost savings)

**How to connect Power BI / Tableau:**
1. Click your SQL Warehouse → **Connection details** tab
2. Copy **Server hostname** and **HTTP path**
3. In Power BI: Get Data → Azure → Azure Databricks
4. Paste Server hostname + HTTP path → sign in via Azure AD

> **Key difference:** Notebook cluster = for Data Engineers (Python/PySpark/Scala).  
> SQL Warehouse = for SQL Analysts and BI Tools. Do not attach a notebook to a SQL Warehouse!


### Cluster Info

Check the runtime version and Photon status on the current cluster.

In [0]:
# Cluster runtime and Photon status
dbr_version = spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion", "unknown")
photon = spark.conf.get("spark.databricks.photon.enabled", "false")
print(f"Runtime: {dbr_version}  |  Photon: {photon}")

### Monitoring

Where to look for problems: **Cluster → Event log** | **Spark UI** (Jobs, SQL tabs) | **Driver/Executor logs**

> **Best Practice:** For production pipelines, log to Delta tables — not just cluster logs.

## Magic Commands

Magic commands allow you to switch between languages and perform system operations directly from notebook cells.

---

| Command | Purpose |
|---------|---------|
| `%sql` | SQL cell |
| `%python` | Python cell (default) |
| `%md` | Markdown documentation |
| `%fs` | DBFS file operations |
| `%sh` | Shell commands |
| `%run` | Execute another notebook |
| `%pip` | Install notebook-scoped libraries |

## Cost Management

Understanding how costs are generated in Databricks is essential from day one. Costs are driven by **Databricks Units (DBU)** — a normalised unit of processing capability per hour. You pay for DBUs × the VM cost per hour.

---

### What is a DBU?

A **Databricks Unit (DBU)** is Databricks' internal measure of compute consumption:
- Every cluster consumes a certain number of DBUs per hour depending on instance type, tier (Standard / Premium / Enterprise), and feature (Photon, ML, SQL Classic/Pro/Serverless).
- You are billed: `DBU rate × hours running × number of nodes`.
- DBU rates differ by compute type — Job clusters always cost less per DBU than All-Purpose clusters.

### Cluster Types and Their Cost Impact

| Cluster Type | Use Case | DBU Rate | Auto-terminate |
|---|---|---|---|
| **All-Purpose** | Interactive notebooks, development | Higher (charged per hour running) | Optional — **always set it!** |
| **Job Cluster** | Scheduled workflows, production pipelines | Lower (charged only during job run) | Always — terminates when job ends |
| **SQL Warehouse Classic** | BI queries, SQL analytics | Medium | Yes — after idle period |
| **SQL Warehouse Serverless** | Interactive SQL, dashboards | Higher/query but zero idle cost | Instant — stops between queries |
| **Serverless Notebooks** | Development with fast startup | Pay-per-execution, no idle cost | Automatic — session-scoped |

> **Rule of thumb:** For scheduled, automated pipelines always use a **Job cluster** — it starts for the job and stops immediately when done. All-Purpose clusters keep running even when idle.

---

### Autoscaling

Autoscaling lets Databricks automatically adjust the number of worker nodes based on workload:

- **Enabled:** Cluster scales from `min_workers` to `max_workers` dynamically. Efficient for variable workloads.
- **Disabled:** Fixed number of workers. Predictable cost, faster start.

> ⚠️ **Autoscaling caveat:** With autoscaling enabled, costs can spike unexpectedly if a workload is large. Always set a reasonable `max_workers` limit and configure a **Budget Alert** in the Account Console.

---

### Cluster Policies — Enforce Cost Guardrails

A **Cluster Policy** is a JSON template that restricts what users can configure when creating a cluster:

```json
{
  "autotermination_minutes": { "type": "fixed", "value": 30 },
  "num_workers":             { "type": "range",  "maxValue": 8 },
  "node_type_id":            { "type": "allowlist", "values": ["Standard_DS3_v2"] }
}
```

Key policy controls:
- **`autotermination_minutes`** — force idle termination (e.g. 30 min)
- **`num_workers` max** — cap the cluster size unexperienced users can spin up
- **`node_type_id` allowlist** — restrict to cost-effective instance types
- **`spark_version` allowlist** — avoid users running expensive GPU-enabled runtimes by mistake

> Policies are configured in **Workspace → Compute → Cluster Policies**. Assign a policy to a user group via the Admin Console.

---

### Cost Tags — Chargeback & Attribution

Every cluster and SQL Warehouse can have **custom tags** (`key: value`) that appear in the billing export:

```python
# Set via cluster creation API or UI → Advanced Options → Tags
tags = {
    "project":     "retail_pipeline",
    "team":        "data_engineering",
    "environment": "production",
    "cost_center": "CC-1042"
}
```

Tags flow through to `system.billing.usage.custom_tags` allowing you to slice costs by project, team, or environment in SQL queries.

---

### Practical Cost Checklist

| Action | Saving |
|--------|--------|
| Set auto-termination (30 min) on All-Purpose clusters | ★★★★ |
| Use Job clusters for scheduled pipelines | ★★★★ |
| Enable Spot/Preemptible for worker nodes | ★★★ |
| Use Serverless for notebooks (no idle time) | ★★★ |
| Right-size: start with 2-4 workers, scale if needed | ★★★ |
| Enforce Cluster Policies for all teams | ★★★ |
| Tag all resources for chargeback reporting | ★★ |
| Enable Photon only for heavy aggregations/joins | ★★ |

---

### Cost Monitoring: Where to Look

| Tool | What it shows |
|------|--------------|
| **Workspace → Compute** | Running clusters, uptime, instance type |
| **Account Console → Cost Analysis** | DBU usage by workspace, cluster, user — visual |
| **`system.billing.usage`** (Unity Catalog) | Full DBU log queryable via SQL — most powerful |
| **Budget Alerts** (Account Console) | E-mail / webhook when spending exceeds a threshold |
| **[Azure Pricing Calculator](https://azure.microsoft.com/en-us/pricing/calculator/?service=databricks)** | Estimate monthly cost before provisioning — select region, tier, instance type |
| **[Databricks DBU Pricing (Azure)](https://www.databricks.com/product/pricing/databricks-on-azure)** | Official DBU rates per SKU — All-Purpose, Jobs, SQL, Serverless |

> **`system.billing.usage`** is available in every Unity Catalog workspace. It contains one row per billing record with columns for workspace, cluster, user, DBU count, SKU, and custom tags. The cells below show how to query it.

---

### DBU Usage Report — `system.billing.usage`

The `system.billing.usage` table is the authoritative source for programmatic cost analysis. It is updated daily and lives in the `system` catalog, accessible to workspace admins and users granted the `SELECT` privilege.

**Key columns:**

| Column | Description |
|--------|-------------|
| `workspace_id` | Databricks workspace ID |
| `sku_name` | Compute SKU, e.g. `STANDARD_ALL_PURPOSE_COMPUTE` |
| `cluster_id` | Cluster that consumed DBUs |
| `usage_start_time` | Start of the billing period (hourly granularity) |
| `usage_quantity` | DBUs consumed in that period |
| `usage_unit` | Always `DBU` |
| `custom_tags` | Map of custom cluster/job tags |
| `usage_metadata.job_id` | Job ID if usage was from a workflow |
| `usage_metadata.notebook_id` | Notebook ID for notebook-driven usage |

> **Requirement:** The `system` catalog must be enabled in your workspace (Account Console → Metastore → System Schemas). In most workspaces it is on by default since DBR 12+.

In [ ]:
# ── DBU Report 1: Last 30 days — total DBU by SKU ─────────────────────
# system.billing.usage is updated daily; requires Unity Catalog.
# If the table is not available, a clear error message is shown.

try:
    dbu_by_sku = spark.sql("""
        SELECT
            sku_name,
            ROUND(SUM(usage_quantity), 2)   AS total_dbu,
            COUNT(*)                         AS billing_records,
            DATE(MIN(usage_start_time))      AS first_day,
            DATE(MAX(usage_start_time))      AS last_day
        FROM system.billing.usage
        WHERE usage_start_time >= CURRENT_DATE - INTERVAL 30 DAYS
          AND usage_unit = 'DBU'
        GROUP BY sku_name
        ORDER BY total_dbu DESC
    """)
    print("DBU consumption by SKU — last 30 days")
    display(dbu_by_sku)
except Exception as e:
    print(f"system.billing.usage not available in this workspace: {e}")
    print("Enable System Schemas in Account Console → Metastore → System Schemas")

In [ ]:
# ── DBU Report 2: Daily DBU trend — last 14 days ──────────────────────
# Run this cell and click "+" → Bar/Line Chart to visualise the trend.

try:
    dbu_daily = spark.sql("""
        SELECT
            DATE(usage_start_time)          AS usage_date,
            sku_name,
            ROUND(SUM(usage_quantity), 2)   AS daily_dbu
        FROM system.billing.usage
        WHERE usage_start_time >= CURRENT_DATE - INTERVAL 14 DAYS
          AND usage_unit = 'DBU'
        GROUP BY DATE(usage_start_time), sku_name
        ORDER BY usage_date ASC
    """)
    print("Daily DBU trend — last 14 days  |  Tip: click '+' → Line Chart → X: usage_date, Y: daily_dbu, Group: sku_name")
    display(dbu_daily)
except Exception as e:
    print(f"system.billing.usage not available: {e}")

In [ ]:
# ── DBU Report 3: Top 10 clusters by DBU consumption (last 30 days) ───

try:
    dbu_by_cluster = spark.sql("""
        SELECT
            cluster_id,
            COALESCE(usage_metadata.cluster_name, cluster_id) AS cluster_name,
            sku_name,
            ROUND(SUM(usage_quantity), 2)                     AS total_dbu,
            COUNT(DISTINCT DATE(usage_start_time))            AS active_days
        FROM system.billing.usage
        WHERE usage_start_time >= CURRENT_DATE - INTERVAL 30 DAYS
          AND usage_unit = 'DBU'
          AND cluster_id IS NOT NULL
        GROUP BY cluster_id, usage_metadata.cluster_name, sku_name
        ORDER BY total_dbu DESC
        LIMIT 10
    """)
    print("Top 10 clusters by DBU consumption — last 30 days")
    display(dbu_by_cluster)
except Exception as e:
    print(f"system.billing.usage not available: {e}")

In [ ]:
# ── DBU Report 4: Cost attribution via custom tags ────────────────────
# Shows DBU consumption broken down by custom cluster tags.
# Useful for chargeback reports — requires clusters to have tags set.
# Replace 'project' with whichever tag key your organisation uses.

TAG_KEY = "project"   # ← change to your tag key, e.g. "team", "cost_center"

try:
    dbu_by_tag = spark.sql(f"""
        SELECT
            custom_tags['{TAG_KEY}']          AS tag_value,
            sku_name,
            ROUND(SUM(usage_quantity), 2)     AS total_dbu,
            COUNT(DISTINCT cluster_id)        AS cluster_count,
            COUNT(DISTINCT DATE(usage_start_time)) AS active_days
        FROM system.billing.usage
        WHERE usage_start_time >= CURRENT_DATE - INTERVAL 30 DAYS
          AND usage_unit = 'DBU'
          AND custom_tags['{TAG_KEY}'] IS NOT NULL
        GROUP BY custom_tags['{TAG_KEY}'], sku_name
        ORDER BY total_dbu DESC
    """)

    if dbu_by_tag.count() == 0:
        print(f"No clusters with tag '{TAG_KEY}' found in the last 30 days.")
        print("Set custom tags on your clusters: Cluster → Edit → Advanced → Tags")
    else:
        print(f"DBU consumption by tag '{TAG_KEY}' — last 30 days")
        display(dbu_by_tag)
except Exception as e:
    print(f"system.billing.usage not available: {e}")

### Demo: %sql

In [0]:
%sql
-- SQL magic command allows writing pure SQL without Python wrapper

SELECT 
  current_catalog() as catalog,
  current_schema() as schema,
  current_user() as user,
  current_timestamp() as timestamp

### Demo: Mixing Python + SQL

Create data in Python → query with SQL via temp view:

In [0]:
# Python: Raw data definition
data = [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob", "Sales", 75000),
    (3, "Charlie", "Engineering", 105000),
    (4, "Diana", "Marketing", 68000),
    (5, "Eve", "Engineering", 98000)
]

# Schema definition
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False)
])

In [0]:
# Create DataFrame
df = spark.createDataFrame(data, schema)
display(df)

In [0]:
# Register as temp view for SQL access
df.createOrReplaceTempView("employees_temp")

In [0]:
%sql
-- SQL: Aggregation on Python data

SELECT 
  department,
  COUNT(*) as employee_count,
  AVG(salary) as avg_salary,
  MAX(salary) as max_salary
FROM employees_temp
GROUP BY department
ORDER BY avg_salary DESC

### Demo: %pip (notebook-scoped libraries)

In [0]:
# Install emoji library
%pip install emoji

In [0]:
import emoji
print(emoji.emojize('Databricks is :fire:'))

### Databricks Assistant (AI)

In 2025, coding work is assisted by AI. Databricks has a built-in assistant (**Databricks Assistant**) that is context-aware of your data (knows table schemas in Unity Catalog!).

**How to use?**
1. Shortcut **Cmd+I** (Mac) or **Ctrl+I** (Windows) inside a cell.
2. "Assistant" side panel.

**What is it for?**
- **Code Generation**: "Write a SQL query that calculates average sales by region from the sales table".
- **Code Explanation**: Select a complex snippet and ask "Explain this code".
- **Fixing errors**: When a cell returns an error, click "Diagnose Error" – the assistant will explain the cause and propose a fix.
- **Transformation**: "Rewrite this code from PySpark to SQL".

---

## Unity Catalog

A modern metadata management system replacing Hive Metastore. Provides centralized access control, lineage, and governance across the entire organization.

---

### Theoretical Introduction

Databricks supports two metadata systems: legacy Hive Metastore and modern Unity Catalog. Unity Catalog is recommended for all new projects due to advanced governance and security features.

**Key Differences:**

| Aspect | Hive Metastore | Unity Catalog |
|--------|----------------|---------------|
| **Governance** | Limited | Full: RBAC, masking, audit |
| **Namespace** | 2-level (db.table) | 3-level (catalog.schema.table) |
| **Cross-workspace** | No | Yes (shared metastore) |
| **Lineage** | None | End-to-end lineage |
| **Data Sharing** | Limited | Delta Sharing protocol |
| **Isolation** | Workspace-level | Catalog-level |

**Why Unity Catalog?**
- Central access management for all workspaces
- Automatic lineage for audit and compliance
- Fine-grained permissions (column-level, row-level)
- Integration with external systems (Delta Sharing)

### Namespace - Hive vs Unity Catalog

#### Example: Namespace Comparison

**Objective:** Compare table access syntax

### Creating a Table in Unity Catalog

#### Example: Creating a Table

**Objective:** Demonstrate full syntax with 3-level namespace

In [0]:
# Table name in Unity Catalog (3-level namespace)
table_name = f"{CATALOG}.{BRONZE_SCHEMA}.retail_sales"

# Sales dataset: 5 regions × 4 categories — good for visualizations
sales_data = [
    ("North",   "Electronics",  142500, 1180, 120.76),
    ("North",   "Apparel",       58300,  940,  62.02),
    ("North",   "Food",          34200,  870,  39.31),
    ("North",   "Home",          71400,  620, 115.16),
    ("South",   "Electronics",   98700,  820, 120.37),
    ("South",   "Apparel",       45100,  730,  61.78),
    ("South",   "Food",          61800, 1540,  40.13),
    ("South",   "Home",          53200,  470, 113.19),
    ("East",    "Electronics",  175300, 1450, 120.90),
    ("East",    "Apparel",       67900, 1090,  62.29),
    ("East",    "Food",          48500, 1210,  40.08),
    ("East",    "Home",          89100,  780, 114.23),
    ("West",    "Electronics",  210400, 1740, 120.92),
    ("West",    "Apparel",       82600, 1330,  62.11),
    ("West",    "Food",          55900, 1380,  40.51),
    ("West",    "Home",         103700,  910, 113.96),
    ("Central", "Electronics",   76300,  630, 121.11),
    ("Central", "Apparel",       31400,  510,  61.57),
    ("Central", "Food",          28700,  710,  40.42),
    ("Central", "Home",          42800,  380, 112.63),
]

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

sales_schema = StructType([
    StructField("region",          StringType(),  False),
    StructField("category",        StringType(),  False),
    StructField("revenue",         IntegerType(), False),
    StructField("transactions",    IntegerType(), False),
    StructField("avg_order_value", DoubleType(),  False),
])

sales_df = spark.createDataFrame(sales_data, sales_schema)
display(sales_df)

In [0]:
# Save as Delta Table in Unity Catalog (3-level namespace)
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"Table saved: {table_name}")

In [0]:
# Read back from Delta — verification
display(spark.table(table_name))

---
## GUI Visualizations — Edit Chart (no code!)

One of the biggest advantages of Databricks is that you can create a chart **with a single click, without any code**.

### How to do it:
1. Run any cell with `display(df)`
2. In the result widget, click **`+`** (next to the "Table" tab)
3. Choose the **chart type**: Bar, Line, Scatter, Pie, Box, Heatmap, Area
4. Set axes, grouping, aggregation
5. Click **Save** — the chart is permanently saved in the cell

> **Tip:** Works for PySpark DataFrames and SQL results (`%sql SELECT ...`)

In [ ]:
# Example 1 — Revenue by region
# After running: click "+" → Bar Chart → X: region, Y: revenue, Aggregation: SUM

from pyspark.sql.functions import sum as spark_sum, desc

display(
    spark.table(table_name)
    .groupBy("region")
    .agg(spark_sum("revenue").alias("total_revenue"))
    .orderBy(desc("total_revenue"))
)

In [ ]:
# Example 2 — Revenue by category
# After running: click "+" → Pie Chart → Key: category, Value: total_revenue

display(
    spark.table(table_name)
    .groupBy("category")
    .agg(spark_sum("revenue").alias("total_revenue"))
    .orderBy(desc("total_revenue"))
)

### Data Profiling (no code!)

Databricks automatically generates a **data profile** — summary statistics for every column.

#### How to do it:
1. Run the cell below with `display(df)` on the full table
2. In the result widget, click **`+`** → **Data Profile**
3. View: min/max, null counts, value distribution per column

> **Useful before analysis** — quickly spot nulls, outliers, unexpected distributions.

In [ ]:
# Full table — click "+" → Data Profile to see statistics for all columns
display(spark.table(table_name))

### Task: Check in UI

1. Click **Catalog** in the left sidebar.
2. Find your catalog (name in `CATALOG` variable, e.g., `retailhub_...`).
3. Expand the `bronze` schema.
4. Click on the `retail_sales` table.
5. See tabs: **Sample Data** (preview) and **Lineage** (data origin).

**Explanation:**

The table was created with a full 3-level namespace. In Unity Catalog, every table automatically:
- Is managed by the governance system
- Has tracked lineage
- Has permissions assigned based on catalog and schema
- Is available in Catalog Explorer for exploration

**Managed vs External Tables:**
The table above is a **Managed Table**. Databricks manages both metadata and data files (in default catalog/schema storage). Dropping the table (`DROP TABLE`) also deletes the data.

**External Table** is created when we provide `LOCATION 'path'`. Then `DROP TABLE` removes only metadata, and files remain in storage.

### Comparison PySpark vs SQL

**DataFrame API (PySpark):**

In [0]:
# PySpark Approach - programmatic DataFrame API
df_pyspark = spark.table(table_name)

In [0]:
result_pyspark = df_pyspark \
    .filter(F.col("region") == "West") \
    .select("region", "category", "revenue", "transactions") \
    .orderBy(F.desc("revenue"))
display(result_pyspark)

**SQL Equivalent:**

In [0]:
df = spark.sql(f"SELECT * FROM {table_name} WHERE region = 'West'")
display(df)

### Parameterization with Databricks Widgets

Below we use the **Widgets** mechanism, which allows creating interactive controls in the notebook. This allows passing parameters (e.g., table names, dates) to SQL and Python code, facilitating the building of universal reports.

In [0]:
# Parameterization with Databricks Widgets
# Set default values based on variables from 00_setup (if available)
# This ensures SQL cells will use the same catalog as Python cells

default_catalog = CATALOG if 'CATALOG' in locals() else "retailhub_trainer"
default_schema = BRONZE_SCHEMA if 'BRONZE_SCHEMA' in locals() else "bronze"

dbutils.widgets.text("CATALOG", default_catalog)
dbutils.widgets.text("BRONZE_SCHEMA", default_schema)
dbutils.widgets.text("BRONZE_SCHEMA_2", default_schema)

In [0]:
# Example of variable injection in SQL using Python f-string
query = f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.retail_sales WHERE region = 'North'"
df = spark.sql(query)
display(df)

In [0]:
%sql
SELECT 
  region,
  category,
  revenue,
  transactions
FROM IDENTIFIER(:CATALOG || '.' || :BRONZE_SCHEMA || '.retail_sales')
WHERE region = 'North'
ORDER BY revenue DESC

**Comparison:**
- **Performance**: Identical - both approaches compile to the same Catalyst query plan
- **When to use PySpark**: 
  - Complex business logic with UDFs
  - Dynamic pipelines (parameterization, loops)
  - Integration with Python libraries (pandas, scikit-learn)
- **When to use SQL**: 
  - Simple transformations and aggregations
  - Team with strong SQL skills
  - Migration from traditional Data Warehouse
  - Better support for business analysts



## Summary

### What was achieved:
- Learned Lakehouse concept as evolution of Data Lake + Data Warehouse
- Explored Databricks platform elements: Workspace, Compute, Catalog
- Understood Unity Catalog hierarchy: Metastore → Catalog → Schema → Objects
- Practiced magic commands: %sql, %python, %fs, %pip
- Compared Hive Metastore vs Unity Catalog
- Created first Delta table in Unity Catalog with 3-level namespace

### Key Takeaways:
1. **Lakehouse eliminates data duplication**: Single copy serves BI, ML, and real-time analytics
2. **Unity Catalog is governance foundation**: 3-level namespace, fine-grained permissions, automatic lineage
3. **Clusters are flexible**: Autoscaling and spot instances reduce costs, Photon accelerates queries
4. **Notebooks are powerful**: Mixing SQL/Python, magic commands, Git integration via Git Folders
5. **Delta Lake is default format**: ACID transactions, time travel, schema evolution

### Quick Reference - Key Commands:

| Operation | PySpark | SQL |
|-----------|---------|-----|
| Set catalog | `spark.sql(f"USE CATALOG {CATALOG}")` | `USE CATALOG my_catalog` |
| List catalogs | `spark.sql("SHOW CATALOGS")` | `SHOW CATALOGS` |
| List schemas | `spark.sql("SHOW SCHEMAS")` | `SHOW SCHEMAS` |
| Create table | `df.write.saveAsTable("cat.schema.table")` | `CREATE TABLE cat.schema.table AS SELECT ...` |
| Read table | `spark.table("cat.schema.table")` | `SELECT * FROM cat.schema.table` |
| Metadata | - | `SELECT * FROM system.information_schema.tables` |
| Install lib | `%pip install package` | - |